In [2]:
import pandas as pd
import numpy as np
jobs=pd.read_csv('data/fake_job_postings.csv')

In [1]:
from codecarbon import EmissionsTracker

/Users/ianch/miniconda3/envs/tf_m4/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [4]:
tracker = EmissionsTracker(
    output_dir='codecarbon_mac/',
    output_file='mac_emissions.csv')

[codecarbon WARNING @ 09:06:01] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 09:06:01] Error while trying to count physical CPUs: [Errno 2] No such file or directory: 'lscpu'. Defaulting to 1.
[codecarbon INFO @ 09:06:01] [setup] RAM Tracking...
[codecarbon INFO @ 09:06:01] [setup] CPU Tracking...
[codecarbon WARNING @ 09:06:01] We saw that you have a Apple M4 but we don't know it. Please contact us.
[codecarbon WARNING @ 09:06:01] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Mac OS and ARM processor detected: Please enable PowerMetrics sudo to measure CPU

[codecarbon INFO @ 09:06:01] CPU Model on constant consumption mode: Apple M4
[codecarbon WARNING @ 09:06:01] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 09:06:01] [setup] GPU Tracking...
[codecarbon INFO @ 09:06:01] No GPU found.
[codecarbon INFO @ 09:06:01] The below tracking methods have been set up:
            

In [2]:
jobs['fraudulent'].value_counts()

fraudulent
0    17014
1      866
Name: count, dtype: int64

In [3]:
indices_to_drop = jobs[jobs['fraudulent'] == 0].sample(n=16000).index #fixing class imbalance
jobs = jobs.drop(indices_to_drop)

In [4]:
jobs['fraudulent'].value_counts()

fraudulent
0    1014
1     866
Name: count, dtype: int64

**Commitments made in the project plan**

1. Methods: Logistic Regression, SVM, RNN, Random Forest, BERT Transformer
2. Utilize the same or similar pre-processing technique
3. I will then train and test each model, evaluating
accuracy, precision, recall, and the macro-averaged F1 score and utilizing CodeCarbon’s ability to
estimate CO2 Emissions to log and obtain the emissions from training the data and report the calculated
CE_Rel and delta CE_rel metrics.

In [6]:
jobs.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
19,20,Process Controls Engineer - DCS PLC MS Office ...,"US, PA, USA Northeast",NaN,NaN,We Provide Full Time Permanent Positions for m...,Experienced Process Controls Engineer is requi...,Must have 5 or more years of experience with D...,NaN,0,0,0,Full-time,NaN,NaN,NaN,NaN,0
30,31,Customer Service Technical Specialist,"US, MA, Waltham",NaN,NaN,"Novitex Enterprise Solutions, formerly Pitney ...",The Customer Service Technical Specialist will...,Qualifications:Minimum of 6 months customer se...,NaN,0,1,0,Full-time,Entry level,High School or equivalent,Computer Software,Customer Service,0
56,57,Outside Sales Professional-Oronoco,"US, MN, Oronoco",NaN,NaN,"ABC Supply Co., Inc. is the nation’s largest w...","As an Outside Sales Representative, you must h...",Track Record of Sales Success – B2B or B2CNo m...,"As an Outside Sales Representative, you will r...",0,1,0,NaN,NaN,NaN,NaN,NaN,0
58,59,Marketing Guru,"US, TX, Fort Worth",NaN,NaN,We're a Fort Worth based startup trying to cha...,We're a Fort Worth based startup trying to cha...,You've got: 3 - 5 years of technology or healt...,"And for your efforts, we happily provide: A co...",0,1,0,Full-time,Associate,Bachelor's Degree,Computer Software,Marketing,0
87,88,Customer Service Associate - On Call,"US, TN, Franklin",NaN,NaN,"Novitex Enterprise Solutions, formerly Pitney ...",The Customer Service Associate will be based ...,Required Qualifications:High school diploma or...,NaN,0,1,0,Full-time,Entry level,High School or equivalent,Insurance,Customer Service,0


In [5]:
model_df = jobs[['description', 'fraudulent']]

In [6]:
model_df.head()


,description,fraudulent
9,The Customer Service Associate will be based i...,0
18,Kettle is hiring a Visual Designer!Job Locatio...,0
98,"IC&amp;E Technician | Bakersfield, CA Mt. Poso...",1
105,GetCloudServices is a privately held technolog...,0
107,Forward Partners invest in very early stage e-...,0


In [7]:
import nltk
import re
import html
from nltk.corpus import stopwords
nltk.download('stopwords',quiet=True)
stop_words=set(stopwords.words('english'))


def preprocess_text(text):
  text=re.sub(r"(?:http\S+|@)","",text) #arguments are pattern,replace,string.
  text=html.unescape(text) #convert XML to string. This function can handle XML entities like &amp
  tokens=text.split() #just using split()
  tokens=[token for token in tokens if token not in stop_words]
  return ' '.join(tokens)

there's a class imbalance. much more non-fraudulent ones than fraudulent. There are ~17800 observations and maybe 800 fraudulent ones. Maybe consider fixing this (but don't think it has a direct effect on workload, but maybe want things to be realistic).

**LOGISTIC REGRESSION**

In [ ]:
##CHECK LAB 6 FOR K FOLD cross validation applied to logistic regression

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score

In [9]:
model_df['description']=model_df['description'].astype(str)

/var/folders/tl/0pf8lcrd691gwn8392x3hs6r0000gn/T/ipykernel_9003/3832600678.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df['description']=model_df['description'].astype(str)


In [10]:
processed_descriptions=[preprocess_text(doc) for doc in model_df['description']]

In [11]:
labels=model_df['fraudulent']

In [14]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test=train_test_split(processed_descriptions,labels,test_size=0.3,random_state=42)


In [15]:
tfidf=TfidfVectorizer()
X_tfidf = tfidf.fit_transform(x_train)
X_test_tfidf = tfidf.transform(x_test)

In [17]:
reg_classifier=LogisticRegression()
reg_classifier.fit(X_tfidf,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [18]:
y_pred = reg_classifier.predict(X_test_tfidf) #should break to test train

In [19]:
print(classification_report(y_test, y_pred)) #ofc it did perfectly bc i didnt split it into test.train.

              precision    recall  f1-score   support

           0       0.86      0.91      0.88       309
           1       0.88      0.82      0.85       255

    accuracy                           0.87       564
   macro avg       0.87      0.86      0.87       564
weighted avg       0.87      0.87      0.87       564



**SUPPORT VECTOR MACHINE**

In [20]:
from sklearn.model_selection import cross_validate, KFold, GridSearchCV
from sklearn.base import TransformerMixin
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn import metrics
from sklearn.preprocessing import StandardScaler

In [21]:
class SparsetoDense(TransformerMixin):
  def fit(self, x, y = None, **fit_params):
    return self
  def transform(self, x, y=None, **fit_params):
    return x.toarray()

In [22]:
svm_pipe=Pipeline([
    ('densify',SparsetoDense()),
    ('scale', StandardScaler()),
    ('classify',SVC())
])

kernel= ['rbf', 'linear']
C = [0.001, 0.01, 1, 10] #if its running too long will cut down on some of these combos
svm_params = {
    'classify__kernel': kernel,
    'classify__C': C
}

In [23]:
inner_cv=KFold(n_splits=3, shuffle=True, random_state=1)
outer_cv=KFold(n_splits=5, shuffle=True, random_state=1)

grid_SVC=GridSearchCV(svm_pipe, svm_params, cv=inner_cv)

In [24]:
scores=cross_validate(grid_SVC,
                     X=X_tfidf,
                     y=y_train,
                     cv=outer_cv,
                     scoring=['accuracy','f1','precision','recall'],
                     return_estimator=True)

In [25]:
print(scores['test_accuracy'])
print(scores['test_precision'])
print(scores['test_recall'])
print(scores['test_f1'])

[0.82575758 0.85551331 0.84410646 0.81368821 0.82129278]
[0.83035714 0.8852459  0.78125    0.8034188  0.83193277]
[0.775      0.81818182 0.88495575 0.78333333 0.78571429]
[0.80172414 0.8503937  0.82987552 0.79324895 0.80816327]


In [ ]:
#SVM Cross validation ran for 24 mins (wait it was faster locally on m4)

In [26]:
grid_SVC.fit(X_tfidf,y_train)
grid_SVC.best_params_

{'classify__C': 0.001, 'classify__kernel': 'linear'}

In [30]:
#bestmodel as SVC object
#then do bestmodel.fit
#then get y_pred with bestmodel
SVC_model = SVC(kernel='linear', C=0.001)
SVC_model.fit(X_tfidf, y_train)
y_pred = SVC_model.predict(X_test_tfidf)

In [27]:
#get classifciation report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.91      0.88       309
           1       0.88      0.82      0.85       255

    accuracy                           0.87       564
   macro avg       0.87      0.86      0.87       564
weighted avg       0.87      0.87      0.87       564



In [28]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,y_pred))

[[281  28]
 [ 46 209]]


**RECURRENT NEURAL NET (RNN)**

In [29]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[]


In [30]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import metrics
from keras.layers import Dense,Input, GlobalMaxPooling1D, Dropout
from keras.layers import Conv1D, MaxPooling1D, Embedding, LSTM, SimpleRNN
from keras.models import Model, Sequential
from keras.initializers import Constant

In [31]:
MAX_NUM_WORDS = 20000
MAX_SEQUENCE_LENGTH = 300
VALIDATION_SPLIT = 0.2
EMBEDDING_DIM = 100 #going to use GLOVE if we end up doing embeddings

In [32]:
tokenizer = Tokenizer(num_words=MAX_NUM_WORDS)
tokenizer.fit_on_texts(x_train)
train_sequences = tokenizer.texts_to_sequences(x_train)
test_sequences = tokenizer.texts_to_sequences(x_test)
word_index = tokenizer.word_index

In [33]:
trainvalid_data = pad_sequences(train_sequences, maxlen=MAX_SEQUENCE_LENGTH)
test_data = pad_sequences(test_sequences, maxlen=MAX_SEQUENCE_LENGTH)
trainvalid_labels = to_categorical(y_train, num_classes = 2)
test_labels = to_categorical(y_test, num_classes = 2) 

In [44]:
#think the below section should be moved to the top as we prob want to use GLOVE embeddings for everything

In [34]:
#time to split into train and validation
indices = np.arange(trainvalid_data.shape[0])
np.random.shuffle(indices)
trainvalid_data = trainvalid_data[indices]
trainvalid_labels = trainvalid_labels[indices]
num_validation_samples = int(0.2 * trainvalid_data.shape[0])
x_train = trainvalid_data[:-num_validation_samples]
y_train = trainvalid_labels[:-num_validation_samples]
x_val = trainvalid_data[-num_validation_samples:]
y_val = trainvalid_labels[-num_validation_samples:]

In [35]:
glove_dir = '../glove.6B.100d.txt'
glove_index = {}
with open(glove_dir, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        embeddings = np.asarray(values[1:], dtype='float32')
        glove_index[word]=embeddings

In [36]:
#making the embedding matrix
num_words = len(word_index) + 1
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))
for word, i in word_index.items():
    if i > MAX_NUM_WORDS:
        continue
    embedding_vector = glove_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
    

In [37]:
embedding_layer = Embedding(num_words,
                            EMBEDDING_DIM,
                            embeddings_initializer = Constant(embedding_matrix),
                            input_length = MAX_SEQUENCE_LENGTH,
                            trainable=False)


/Users/ianch/miniconda3/envs/tf_m4/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [38]:
rnnmodel = Sequential()
rnnmodel.add(embedding_layer)
rnnmodel.add(LSTM(128, dropout = 0.25))
rnnmodel.add(Dense(2, activation = 'softmax')) #len(labels_index)

rnnmodel.compile(loss='categorical_crossentropy',
                 optimizer = 'Adam',
                 metrics = ['acc', metrics.Precision(), metrics.Recall(), metrics.F1Score(average='macro')])

In [39]:
tf.debugging.set_log_device_placement(True)
rnn_train = rnnmodel.fit(x_train, y_train,
                         batch_size = 16,
                         epochs = 3,
                         validation_data = (x_val, y_val))

Epoch 1/3
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 65ms/step - acc: 0.5594 - f1_score: 0.5506 - loss: 0.6902 - precision: 0.5594 - recall: 0.5594 - val_acc: 0.6046 - val_f1_score: 0.5226 - val_loss: 0.6521 - val_precision: 0.6046 - val_recall: 0.6046
Epoch 2/3
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 64ms/step - acc: 0.6638 - f1_score: 0.6575 - loss: 0.6274 - precision: 0.6638 - recall: 0.6638 - val_acc: 0.6958 - val_f1_score: 0.6785 - val_loss: 0.5959 - val_precision: 0.6958 - val_recall: 0.6958
Epoch 3/3
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - acc: 0.6980 - f1_score: 0.6944 - loss: 0.5701 - precision: 0.6980 - recall: 0.6980 - val_acc: 0.7034 - val_f1_score: 0.6968 - val_loss: 0.5531 - val_precision: 0.7034 - val_recall: 0.7034


In [ ]:
#recurrent_dropout > 0 majorly messed up performance and caused >1 min/step. 
#claude stated this parameter would cause it to fall back to CPU (and even though 
#i didnt see it in the messages, the insane increase in processing time when i removed
#this setting made me suspicious that it was silently falling back to cpu. 

In [40]:
loss, test_acc, test_precision, test_recall, test_f1 = rnnmodel.evaluate(test_data, test_labels)

18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - acc: 0.7074 - f1_score: 0.7025 - loss: 0.5518 - precision: 0.7074 - recall: 0.7074


In [41]:
print(f'Test data accuracy {test_acc} \n',
      f'Test data precision {test_precision} \n',
      f'Test data Recall {test_recall} \n',
      f'Test data F1 Score {test_f1}')

Test data accuracy 0.707446813583374 
 Test data precision 0.707446813583374 
 Test data Recall 0.707446813583374 
 Test data F1 Score 0.7024621963500977


**RANDOM FOREST**

In [97]:
from sklearn.ensemble import RandomForestClassifier

In [114]:
desclist = list(processed_descriptions)
tokenized = []
for doc in desclist:
    sent = doc.lower()
    sent = doc.split()
    tokenized.append(sent)


In [122]:
sentence_embedding_list = []
for sentence in tokenized:
    word_embeddings = [glove_index.get(word) if word in glove_index.keys() else np.zeros(100) for word in sentence]
    if np.sum(word_embeddings)==0:
        sentence_embeddings = np.zeros(100)
    else:
        sentence_embeddings = np.mean(word_embeddings, axis = 0)
    sentence_embedding_list.append(sentence_embeddings)

In [126]:
np.array(sentence_embedding_list) #this should work if the loop ran correctly

array([[-0.12597807,  0.13993321,  0.0185024 , ..., -0.14703711,
         0.44409125,  0.16294534],
       [-0.01549626,  0.08979446,  0.01896285, ..., -0.19430406,
         0.5236805 ,  0.216092  ],
       [-0.044257  ,  0.20607056, -0.08313811, ..., -0.11686583,
         0.40525723,  0.17421933],
       ...,
       [-0.0580673 ,  0.08066092,  0.12535938, ...,  0.1104    ,
         0.21679308,  0.09770462],
       [-0.12449407,  0.11362701, -0.08407349, ..., -0.2438132 ,
         0.54900138,  0.27324951],
       [-0.14515872,  0.09980639, -0.02444845, ..., -0.18500703,
         0.44085038,  0.09799469]])

In [127]:
X_train, X_test, y_train, y_test = train_test_split(sentence_embedding_list, labels, 
                                                    test_size = 0.2,
                                                    random_state=1)

In [128]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [130]:
y_pred = rf.predict(X_test)

In [132]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.88      0.87       217
           1       0.83      0.79      0.81       159

    accuracy                           0.84       376
   macro avg       0.84      0.84      0.84       376
weighted avg       0.84      0.84      0.84       376



**BERT Transformer**

In [35]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from datasets import Dataset

In [38]:
model_df = model_df.reset_index(drop=True)
bert_data = Dataset.from_pandas(model_df)

In [68]:
bert_data = bert_data.rename_column('fraudulent','labels')

In [69]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [70]:
def preprocess_bert(text):
    return tokenizer(text['description'], truncation=True)

In [71]:
tokenized_desc = bert_data.map(preprocess_bert)

Map:   0%|          | 0/1880 [00:00<?, ? examples/s]

In [92]:
tokenized_desc = tokenized_desc.remove_columns(['description'])

In [83]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

In [93]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [94]:
id2label = {0:'REAL', 1:'FRAUDULENT'}
label2id = {'REAL':0, 'FRAUDULENT':1}

In [95]:
bert_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",
                                                               num_labels=2,
                                                               id2label=id2label,
                                                               label2id=label2id)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [96]:
split_dataset = tokenized_desc.train_test_split(test_size=0.2,seed=42)
train_data = split_dataset['train']
test_data=split_dataset['test']

In [97]:
training_args = TrainingArguments(output_dir = 'first_bert_model',
                                  remove_unused_columns = False,
                                 push_to_hub = False)

trainer = Trainer(model=bert_model,
                 args=training_args,
                 train_dataset=train_data,
                 eval_dataset=test_data,
                 processing_class=tokenizer,
                 data_collator=data_collator) #placeholders

In [98]:
trainer.train()

Step,Training Loss


RuntimeError: MPS backend out of memory (MPS allocated: 10.02 GiB, other allocations: 8.03 GiB, max allowed: 18.13 GiB). Tried to allocate 96.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [91]:
train_data


Dataset({
    features: ['description', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1504
})